# Week 2, Stage 1 (pilot) — latent-agent-cost

Timing + first-signal pilot for *Tokens Are Not Dollars*, ahead of the full Week-2 condition matrix. Three conditions on ProsQA only (Qwen2.5-1.5B-Instruct, 500 tasks × seed 0):

- `latent` — training-free Coconut-style hidden-state feedback, 16 latent steps, then explicit answer decoding
- `latent_zero` — zero-latent ablation (same schedule, states zeroed)
- `pause_token` — pause-token control (16 filler-token positions, then explicit CoT)

Purpose: measure per-condition wall-clock per attempt and get a first latent-vs-controls signal before committing to the full matrix (4 conditions × 2 datasets × 500 tasks × 3 seeds — roughly 25–30 GPU-hours, more than one session and close to the 30 h/week free quota).

**Staging is lossless:** the runner appends and flushes every attempt to each run's `.attempts.jsonl` as it completes, so a session cut off mid-run keeps all completed attempts (summary tables are written at each run's end). The full Week-2 matrix follows in later staged commits once this pilot's timing is known — see `RUNBOOK.md`.

**Before running:** notebook Settings -> **Accelerator = GPU** (T4 or P100) and **Internet = ON** (needed for the git clone, the HF dataset downloads, and the model download). Use **Save Version -> Save & Run All (Commit)** so outputs persist.

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible. Set Accelerator = GPU in the notebook settings and re-run.")
print("device:", torch.cuda.get_device_name(0))

Clone the project (fresh every run) and install its dependencies. `requirements.txt` already includes `tokenecon==0.2.0`; the one local pin filtered out is `torch` — Kaggle's preinstalled CUDA build must be kept, never replaced by the local CPU pin.

In [ ]:
!rm -rf latent-agent-cost
!git clone https://github.com/karmendra8386/latent-agent-cost
%cd latent-agent-cost
!grep -v '^torch' requirements.txt > /tmp/req-kaggle.txt
!pip install -q -r /tmp/req-kaggle.txt
!python -c "import torch; print('after install - torch:', torch.__version__, '| cuda:', torch.cuda.is_available())"

**latent / ProsQA** — Latent condition — 16 hidden-state feedback steps, then explicit answer decoding.

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition latent --latent-steps 16 --out results/week2s1_latent_prosqa --label WEEK2S1

**latent_zero / ProsQA** — Zero-latent ablation — same 16-step schedule, fed-back states replaced with zeros (compute-matched; tests whether the states carry task information).

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition latent_zero --latent-steps 16 --out results/week2s1_zero_prosqa --label WEEK2S1

**pause_token / ProsQA** — Pause-token control — 16 filler-token positions processed before explicit CoT: the latent budget spent on visible filler instead of latent states.

In [ ]:
!python scripts/run_baseline.py --dataset prosqa --n 500 --seeds 0 --model Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --condition pause_token --latent-steps 16 --out results/week2s1_pause_prosqa --label WEEK2S1

Show the resulting five-metric tables.

In [ ]:
!ls -la results/
!cat results/week2s1_latent_prosqa.md
!cat results/week2s1_zero_prosqa.md
!cat results/week2s1_pause_prosqa.md